# 01. Study design and data

**Question.** At equal added duration, is lengthening a few vowels harder for a speech recogniser than slowing
the whole utterance?

This notebook documents the test material: which recordings are evaluated, which EveryAyah reciters each
model was allowed to hear, and which evaluated recordings actually occur in the EveryAyah training split.
Everything is read from files in `data/` and `results/`; no numbers are typed by hand.

In [1]:
import sys, json, gzip, collections
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import numpy as np
import stats as S

## Evaluated recordings
`manifest_v2.json` fixes the verses before any recognition: 60 random verse IDs per confirmatory reciter, and the earlier sets for three exploratory reciters.

In [2]:
manifest = json.loads((ROOT / "src/original/confirmation/manifest_v2.json").read_text())
table = {name: (g["role"], len(g["files"])) for name, g in manifest["reciters"].items()}
for name, (role, n) in table.items():
    print(f"{name:11s} {role:18s} {n:3d} candidate recordings")
print("total candidates:", sum(n for _, n in table.values()))

AbdulBasit  exploratory_rerun   35 candidate recordings
Husary      exploratory_rerun   39 candidate recordings
Minshawy    exploratory_rerun   36 candidate recordings
Alafasy     confirmation        60 candidate recordings
Sudais      confirmation        60 candidate recordings
Shuraym     confirmation        60 candidate recordings
Dussary     confirmation        60 candidate recordings
Rifai       confirmation        60 candidate recordings
total candidates: 410


## Retained recordings
A recording is kept when CTC alignment yields at least two target intervals of 60 ms or more. The alignment outcome is logged in every evaluation run; we read it from the Tarteel run.

In [3]:
kept, excluded = collections.Counter(), collections.Counter()
with gzip.open(ROOT / "results/tarteel/observations.jsonl.gz", "rt", encoding="utf-8") as fh:
    for line in fh:
        r = json.loads(line)
        if r["kind"] == "complete": kept[r["reciter"]] += 1
        if r["kind"] == "excluded": excluded[r["reciter"]] += 1
for name in manifest["reciters"]:
    print(f"{name:11s} kept {kept[name]:3d}  excluded {excluded[name]:3d}")
print("kept in total:", sum(kept.values()), "| confirmatory kept:", sum(kept[n] for n in S.CONFIRMATORY))

AbdulBasit  kept  35  excluded   0
Husary      kept  39  excluded   0
Minshawy    kept  36  excluded   0
Alafasy     kept  42  excluded  18
Sudais      kept  42  excluded  18
Shuraym     kept  42  excluded  18
Dussary     kept  42  excluded  18
Rifai       kept  42  excluded  18
kept in total: 320 | confirmatory kept: 210


## Corpus and exposure sets
EveryAyah (Hugging Face revision `6ea5108`) labels every clip with a reciter. The three trained models differ only in which clips of the evaluated reciters they may see.

In [4]:
census = json.loads((ROOT / "data/census_current.json").read_text())
labels = sorted({r for f in census["files"].values() for r in f["reciters"]})
print("revision", census["revision"], "|", len(labels), "reciter labels")
EVALUATED = {"abdul_basit", "husary", "minshawi", "menshawi", "alafasy", "abdurrahmaan_as-sudais",
             "saood_ash-shuraym", "yasser_ad-dussary", "hani_rifai"}          # 8 reciters, 9 labels
HELD_OUT = {"sahl_yassin", "akram_alalaqimy", "muhsin_al_qasim"}              # public benchmark hold-out
print("excluded from M0:", sorted(EVALUATED | HELD_OUT))
eval_texts = json.loads((ROOT / "data/eval_texts.json").read_text())["eval_texts"]
print("candidate verse texts dropped for M1, per label:", {k: len(v) for k, v in eval_texts.items()})

revision 6ea510862d64f59e555a4a8363eebc0f415621df | 36 reciter labels
excluded from M0: ['abdul_basit', 'abdurrahmaan_as-sudais', 'akram_alalaqimy', 'alafasy', 'hani_rifai', 'husary', 'menshawi', 'minshawi', 'muhsin_al_qasim', 'sahl_yassin', 'saood_ash-shuraym', 'yasser_ad-dussary']
candidate verse texts dropped for M1, per label: {'abdul_basit': 29, 'husary': 39, 'minshawi': 29, 'menshawi': 29, 'alafasy': 60, 'abdurrahmaan_as-sudais': 60, 'saood_ash-shuraym': 60, 'yasser_ad-dussary': 60, 'hani_rifai': 60}


## Which evaluated recordings did M2 actually hear?
M2 differs from M1 only by the training clips of the evaluated verse texts. An evaluated recording counts as heard when its reciter's recording of that verse is in the EveryAyah training split.

In [5]:
membership = json.loads((ROOT / "data/split_membership.json").read_text())["study"]
heard = {(n, v) for n, info in membership.items() for v, s in info["per_verse"].items() if "train" in s}
retained = set()
with gzip.open(ROOT / "results/tarteel/observations.jsonl.gz", "rt", encoding="utf-8") as fh:
    for line in fh:
        r = json.loads(line)
        if r["kind"] == "complete": retained.add((r["reciter"], r["verse"]))
conf = {x for x in retained if x[0] in S.CONFIRMATORY}
print("confirmatory recordings:", len(conf), "| heard by M2:", len(conf & heard))
print("all retained recordings:", len(retained), "| heard by M2:", len(retained & heard))

confirmatory recordings: 210 | heard by M2: 167
all retained recordings: 320 | heard by M2: 272
